# TRABALHO FINAL INTEGRADOR - SEMANA 1
## Detecção de Fraude em Transações Financeiras
### Cenário 4: Detecção de Fraude ou Transações Financeiras Suspeitas

---

**Disciplina:** Tópicos Especiais 2 - Agrupamento de Dados e Inteligência Computacional  
**Professores:** Dr. Gustavo Prado e Dr. Clarimundo Machado  
**Data:** Semana 1 de 5

---

## 📋 Divisão de Papéis

| Aluno | Papel | Responsabilidades |
|-------|-------|-------------------|
| **Ruan** | Agrupamento de Dados | Implementação técnica em Python, detecção de outliers, clustering |
| **Lucio** | Dados, Pipeline e Integração | Análise exploratória, documentação, repositório, decisões de dados |
| **Artur** | Inteligência Computacional e Decisão | Modelagem preditiva, técnicas de IC, métricas e avaliação |

---

## 1️⃣ DESCRIÇÃO DO PROBLEMA E DA BASE DE DADOS

### Problema
Instituições financeiras precisam identificar transações suspeitas em bases **altamente desbalanceadas**, nas quais fraudes são extremamente raras (anomalias < 1% dos dados).

### Pergunta Central
**Como usar agrupamento para identificar padrões normais e anômalos e apoiar um modelo de detecção de fraude ou risco?**

### Base de Dados Utilizada
- **Nome:** Credit Card Fraud Detection
- **Fonte:** [Kaggle - MLG ULB Dataset](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)
- **Tamanho:** 284.807 transações
- **Fraudes:** 492 transações (0.173% - altamente desbalanceado)
- **Características:** Dados de cartão de crédito europeu, período de 2 dias em setembro de 2013
- **Variáveis:** 31 colunas (28 features PCA + Amount + Time + Class)

### Relevância do Problema
- Detecção em tempo real de fraude
- Proteção do cliente e instituição
- Desafio de desbalanceamento de classes
- Necessidade de alta precisão e recall

---

## 2️⃣ IMPORTAÇÕES E CONFIGURAÇÕES INICIAIS

In [ ]:
# Importações básicas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Configurações visuais
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Opções de display
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

print("✅ Ambiente configurado com sucesso!")
print(f"Timestamp: {datetime.now()}")

## 3️⃣ CARREGAMENTO DA BASE DE DADOS

**Responsável:** Lucio (com apoio de Ruan)  
**Tarefa:** Carregar e documentar a origem dos dados

In [ ]:
# TODO: Lucio e Ruan - Adaptar o caminho do arquivo conforme necessário
# Baixar em: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
# Descompactar na pasta: dados/

# Carregamento da base
try:
    df = pd.read_csv('creditcard.csv')  # Ajustar caminho se necessário
    print(f"✅ Base carregada com sucesso!")
    print(f"   - Linhas: {df.shape[0]:,}")
    print(f"   - Colunas: {df.shape[1]}")
except FileNotFoundError:
    print("❌ Arquivo não encontrado. Verifique o caminho: creditcard.csv")
    print("   Baixe em: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud")

## 4️⃣ ANÁLISE EXPLORATÓRIA INICIAL (EDA)

**Responsável:** Ruan (Implementação técnica) + Lucio (Interpretação)  
**Tarefas:** Explorar estrutura, tipos de dados, distribuições e padrões iniciais

In [ ]:
# 4.1 - Primeiras linhas
print("="*80)
print("PRIMEIRAS LINHAS DA BASE")
print("="*80)
display(df.head())

In [ ]:
# 4.2 - Informações básicas
print("="*80)
print("INFORMAÇÕES GERAIS")
print("="*80)
print(f"\nForma: {df.shape}")
print(f"\nTipos de dados:")
print(df.dtypes)
print(f"\nMemória utilizada: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
# 4.3 - Estatísticas descritivas
print("="*80)
print("ESTATÍSTICAS DESCRITIVAS")
print("="*80)
display(df.describe().T)

In [ ]:
# 4.4 - Análise da variável alvo (Class - Fraude)
print("="*80)
print("ANÁLISE DA VARIÁVEL ALVO (FRAUDE)")
print("="*80)

class_counts = df['Class'].value_counts()
class_pct = df['Class'].value_counts(normalize=True) * 100

print(f"\nDistribuição:")
print(f"  - Transações legítimas (0): {class_counts[0]:,} ({class_pct[0]:.2f}%)")
print(f"  - Fraudes (1): {class_counts[1]:,} ({class_pct[1]:.2f}%)")
print(f"\n⚠️  IMPORTANTE: Base ALTAMENTE DESBALANCEADA!")
print(f"    Razão: {class_counts[0] / class_counts[1]:.1f} legítimas por 1 fraude")

# Visualização
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

class_counts.plot(kind='bar', ax=axes[0], color=['green', 'red'])
axes[0].set_title('Contagem de Fraudes vs Legítimas', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Classe')
axes[0].set_ylabel('Quantidade')
axes[0].set_xticklabels(['Legítima (0)', 'Fraude (1)'], rotation=0)

class_pct.plot(kind='pie', ax=axes[1], autopct='%1.2f%%', colors=['green', 'red'])
axes[1].set_title('Proporção de Fraudes', fontsize=12, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# 4.5 - Análise da variável Amount (valor da transação)
print("="*80)
print("ANÁLISE DO VALOR DAS TRANSAÇÕES (AMOUNT)")
print("="*80)

print("\nTransações Legítimas:")
print(df[df['Class']==0]['Amount'].describe())

print("\nFraudes:")
print(df[df['Class']==1]['Amount'].describe())

# Visualização
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Distribuição
axes[0].hist([df[df['Class']==0]['Amount'], df[df['Class']==1]['Amount']], 
             label=['Legítima', 'Fraude'], bins=50, color=['green', 'red'], alpha=0.7)
axes[0].set_title('Distribuição do Valor das Transações', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Valor (Amount)')
axes[0].set_ylabel('Frequência')
axes[0].legend()

# Box plot
df.boxplot(column='Amount', by='Class', ax=axes[1])
axes[1].set_title('Box Plot do Valor por Classe', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Classe')
axes[1].set_ylabel('Valor (Amount)')

plt.suptitle('')
plt.tight_layout()
plt.show()

## 5️⃣ TRATAMENTO DE VALORES AUSENTES

**Responsável:** Ruan  
**Tarefa:** Identificar e tratar valores ausentes (NaN, None)

In [ ]:
# 5.1 - Identificar valores ausentes
print("="*80)
print("ANÁLISE DE VALORES AUSENTES")
print("="*80)

missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Coluna': missing.index,
    'Valores Ausentes': missing.values,
    'Percentual (%)': missing_pct.values
}).sort_values('Valores Ausentes', ascending=False)

print(f"\nTotal de colunas: {len(df.columns)}")
print(f"Colunas sem ausentes: {(missing == 0).sum()}")
print(f"Colunas com ausentes: {(missing > 0).sum()}")
print(f"\nDetalhes:\n")
display(missing_df[missing_df['Valores Ausentes'] > 0])

if missing.sum() == 0:
    print("\n✅ Excelente! Não há valores ausentes na base.")

## 6️⃣ TRATAMENTO DE DUPLICIDADES

**Responsável:** Ruan  
**Tarefa:** Identificar e remover registros duplicados

In [ ]:
# 6.1 - Análise de duplicidades
print("="*80)
print("ANÁLISE DE DUPLICIDADES")
print("="*80)

duplicados_total = df.duplicated().sum()
duplicados_partial = df.duplicated(subset=['Time', 'Amount']).sum()

print(f"\nLinhas completamente duplicadas: {duplicados_total}")
print(f"Linhas duplicadas (Time + Amount): {duplicados_partial}")

if duplicados_total > 0:
    print(f"\n⚠️  Removendo {duplicados_total} linhas duplicadas...")
    df = df.drop_duplicates()
    print(f"✅ Novo tamanho da base: {df.shape[0]:,} linhas")
else:
    print(f"\n✅ Nenhuma duplicidade encontrada.")

## 7️⃣ IDENTIFICAÇÃO DE OUTLIERS

**Responsável:** Ruan (Implementação) + Artur (Análise)  
**Tarefa:** Detectar anomalias e outliers (crítico para detecção de fraude!)

In [ ]:
# 7.1 - Método IQR (Interquartile Range)
print("="*80)
print("DETECÇÃO DE OUTLIERS - MÉTODO IQR")
print("="*80)

def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return (data[column] < lower_bound) | (data[column] > upper_bound)

# Aplicar a diferentes colunas
outlier_cols = ['Amount', 'Time']
outlier_summary = {}

for col in outlier_cols:
    outliers = detect_outliers_iqr(df, col)
    outlier_count = outliers.sum()
    outlier_pct = (outlier_count / len(df)) * 100
    outlier_summary[col] = {'count': outlier_count, 'pct': outlier_pct}
    print(f"\n{col}:")
    print(f"  - Outliers detectados: {outlier_count} ({outlier_pct:.2f}%)")

print("\n⚠️  IMPORTANTE: Outliers em Amount podem ser transações legítimas!")
print("    NÃO remover agora - importante para análise de fraude posterior.")

In [ ]:
# 7.2 - Visualização de outliers no Amount
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot geral
bp = axes[0].boxplot(df['Amount'], vert=True)
axes[0].set_title('Box Plot do Amount (Geral)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Valor (Amount)')

# Box plot por classe
data_to_plot = [df[df['Class']==0]['Amount'], df[df['Class']==1]['Amount']]
bp2 = axes[1].boxplot(data_to_plot, labels=['Legítima', 'Fraude'])
axes[1].set_title('Box Plot do Amount (por Classe)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Valor (Amount)')

plt.tight_layout()
plt.show()

## 8️⃣ TRANSFORMAÇÃO DE VARIÁVEIS CATEGÓRICAS

**Responsável:** Ruan  
**Tarefa:** Analisar e transformar variáveis categóricas (se houver)

In [ ]:
# 8.1 - Identificar variáveis categóricas
print("="*80)
print("ANÁLISE DE VARIÁVEIS CATEGÓRICAS")
print("="*80)

categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"\nVariáveis categóricas encontradas: {categorical_cols}")

if len(categorical_cols) == 0:
    print("✅ Nenhuma variável categórica encontrada.")
    print("   A base já está numérica (features foram transformadas por PCA).")
else:
    print("\nColunas categóricas e seus valores únicos:")
    for col in categorical_cols:
        print(f"  - {col}: {df[col].nunique()} valores únicos")
        print(f"    Valores: {df[col].unique()[:10]}")

## 9️⃣ DESCRIÇÃO DOS ATRIBUTOS

**Responsável:** Lucio  
**Tarefa:** Documentar e descrever cada atributo

In [ ]:
print("="*80)
print("DESCRIÇÃO DOS ATRIBUTOS")
print("="*80)

attributes_description = """
ATRIBUTOS DA BASE:

1. Time (int)
   - Segundos decorridos entre esta transação e a primeira da base
   - Intervalo: 0 a 172792 segundos (~2 dias)

2. V1 a V28 (float)
   - Componentes principais resultados de transformação PCA
   - Mantidas por confidencialidade (não há interpretação direta)
   - Escaladas e normalizadas
   - Estas são as PRINCIPAIS FEATURES para análise

3. Amount (float)
   - Valor da transação em dólar
   - Intervalo: $0.00 a $25,691.16
   - NÃO foi transformado por PCA

4. Class (int) - VARIÁVEL ALVO
   - 0 = Transação legítima
   - 1 = Fraude
   - DESBALANCEADA: 99.83% legítimas, 0.17% fraudes
"""

print(attributes_description)

## 🔟 CORRELAÇÕES E PADRÕES

**Responsável:** Ruan (Implementação) + Lucio (Análise)  
**Tarefa:** Explorar correlações entre variáveis

In [ ]:
# 10.1 - Correlação com a variável alvo
print("="*80)
print("CORRELAÇÃO COM FRAUDE (Class)")
print("="*80)

correlation_with_class = df.corr()['Class'].sort_values(ascending=False)
print("\nTop 15 features mais correlacionadas com fraude:")
print(correlation_with_class.head(15))

# Visualização
plt.figure(figsize=(10, 8))
correlation_with_class[1:16].plot(kind='barh')
plt.title('Top 15 Features Correlacionadas com Fraude', fontsize=12, fontweight='bold')
plt.xlabel('Correlação')
plt.tight_layout()
plt.show()

In [ ]:
# 10.2 - Matriz de correlação (amostra)
print("\n" + "="*80)
print("MATRIZ DE CORRELAÇÃO - TOP FEATURES")
print("="*80)

top_features = correlation_with_class[1:6].index.tolist() + ['Amount', 'Class']
corr_matrix = df[top_features].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlação entre Top Features e Amount', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 1️⃣1️⃣ STATUS FINAL E PRÓXIMAS ETAPAS

**Responsável:** Lucio (Coordenação) + Ruan (Técnico)

In [ ]:
print("="*80)
print("RESUMO DA SEMANA 1")
print("="*80)

print(f"\n✅ STATUS ATUAL:")
print(f"   - Linhas na base: {df.shape[0]:,}")
print(f"   - Colunas: {df.shape[1]}")
print(f"   - Valores ausentes: {df.isnull().sum().sum()}")
print(f"   - Linhas duplicadas removidas: {duplicados_total}")
print(f"   - Outliers identificados: SIM (não removidos)")
print(f"   - Variáveis categóricas: {len(categorical_cols)}")

print(f"\n📊 CARACTERÍSTICAS IMPORTANTES:")
print(f"   - Base ALTAMENTE DESBALANCEADA: {class_pct[1]:.2f}% fraudes")
print(f"   - Features: 28 componentes PCA + Amount + Time")
print(f"   - Período: ~2 dias (setembro de 2013)")
print(f"   - Valores de transações: ${df['Amount'].min():.2f} a ${df['Amount'].max():.2f}")

print(f"\n🎯 PRÓXIMAS ETAPAS (SEMANA 2-3):")
print(f"   - Normalização/Padronização das features")
print(f"   - Aplicar K-Means")
print(f"   - Aplicar DBSCAN (melhor para anomalias)")
print(f"   - Avaliar clusters com métricas (Silhouette, Davies-Bouldin)")
print(f"   - Interpretar padrões de fraude encontrados")

print(f"\n👥 DIVISÃO DE TRABALHO:")
print(f"   - Ruan: Implementação técnica do preprocessing")
print(f"   - Lucio: Documentação e análise de negócio")
print(f"   - Artur: Planejamento de técnicas de IC e métricas")

## 📝 NOTAS E OBSERVAÇÕES

### Desafios Identificados:
1. **Desbalanceamento extremo:** 0.17% fraudes vs 99.83% legítimas
   - Necessário usar métricas apropriadas (AUC-ROC, F1-score, Precision-Recall)
   - Possível usar técnicas de balanceamento (SMOTE, class weights)

2. **Privacidade dos dados:**
   - Features V1-V28 já transformadas por PCA
   - Não há interpretação direta das variáveis
   - Amount e Time são as únicas interpretáveis

3. **Padrão temporal:**
   - Base cobre apenas ~2 dias
   - Possível sazonalidade em períodos maiores

### Oportunidades para Clustering:
1. Usar DBSCAN para detectar transações anômalas
2. K-Means pode agrupar padrões de comportamento legítimo
3. Clusters podem ser features para modelo de IC

### Referências:
- Dataset: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
- Trabalho original: [Kaggle Competition]